# Group-meeting figures — 2026-08-29 (KL_Aug_28.pptx)

Generates every figure for the group-meeting deck from committed campaign
data (prompt 19). Runs top-to-bottom clean with or without the
maybe-arriving CRC results (`contour_303x317_A`, `placebo`) — those are
auto-detected in the P2 cells and never required.

Execute headlessly:

```bash
source ~/venvs/gm0829/bin/activate
cd trial_0826/campaign/analysis/gm_0829
jupyter nbconvert --to notebook --execute gm_figures.ipynb --output gm_figures.executed.ipynb
```

Outputs: `figs/fig*.png` (300 dpi) + `figs/fig*.pdf`, and a regenerated
`FIGURES.md` (captions stay in sync with whatever data was present at run
time).


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- paths (robust to cwd = campaign/ or campaign/analysis/gm_0829/) ---
_here = Path.cwd().resolve()
CAMP = next(p for p in [_here, *_here.parents] if (p / "waves" / "screening").is_dir())
GM = CAMP / "analysis" / "gm_0829"
FIGS = GM / "figs"
FIGS.mkdir(parents=True, exist_ok=True)

# --- pinned base-case constants (recompute nothing) ---
BASE_CURT_MWH = 1_688_575.777   # total curtailment, base case, MWh/yr
BASE_SHED_MWH = 41_500.115      # total load shed, base case, MWh/yr

# --- publication style ---
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "axes.labelsize": 14,
    "axes.labelweight": "bold",
    "xtick.labelsize": 11.5,
    "ytick.labelsize": 11.5,
    "legend.fontsize": 11.5,
    "legend.frameon": True,
    "font.size": 12,
})
FIGSIZE = (10, 5.6)


def save(fig, name):
    fig.savefig(FIGS / f"{name}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIGS / f"{name}.pdf", dpi=300, bbox_inches="tight")
    print(f"saved figs/{name}.png + .pdf")


# --- load the two frozen full-year datasets ---
scr = pd.read_csv(CAMP / "waves" / "screening" / "objectives.csv")
pil = pd.read_csv(CAMP / "waves" / "pilot" / "objectives.csv")

solo = scr[scr["oat_site"] != "__ALL__"].copy()
allrow = scr[scr["oat_site"] == "__ALL__"].iloc[0]
pilot = pil[pil["num_days"] == 366].iloc[0]  # rows 1-3 bit-identical; use row 1

# verify the pinned numbers against the CSVs, then trust them
_pin = {"121_NUCLEAR_1": -371_141, "303_WIND_1": -671_602,
        "317_WIND_1": -498_712, "122_WIND_1": -416_556}
for site, val in _pin.items():
    got = solo.loc[solo["oat_site"] == site, "delta_curtailment_mwh"].iloc[0]
    assert abs(got - val) < 1.0, (site, got, val)
assert abs(solo["delta_curtailment_mwh"].sum() - (-2_273_490)) < 1.0
assert abs(allrow["delta_curtailment_mwh"] - (-1_470_501)) < 1.0
assert abs(pilot["delta_curtailment_mwh"] - (-1_498_818)) < 1.0
assert abs(pilot["delta_cost_less_synthetic_usd_APPROX"] - 25_597_655) < 1.0
assert abs(allrow["delta_cost_less_synthetic_usd_APPROX"] - 74_405_030) < 1e4
_rep = pil[pil["num_days"] == 366]["total_cost_raw_usd"]
assert _rep.nunique() == 1 and len(_rep) == 3  # determinism fact for fig4
PILOT_REPEAT_COST = _rep.iloc[0]
print("pinned numbers verified; pilot repeat cost =", PILOT_REPEAT_COST)


## P0 · fig1 — "Cleanup is not free, and design sets the price"

Full-year designs on the (curtailment removed, Δ system cost) plane.
Target slide: right after the −89% hook (slide 2/3). Cost is the
synthetic-bid-corrected **approximation** (`_APPROX`).


In [ ]:
WIND_BIG = {"303_WIND_1", "317_WIND_1", "122_WIND_1"}
CLASS_STYLE = {  # consistent meaning across figures
    "Wind, solo (ω=0.5, B=$40)":    dict(color="tab:blue",   marker="o"),
    "PV + tail, solo (ω=0.5, B=$40)": dict(color="tab:orange", marker="^"),
    "Nuclear, solo (ω=0.5, B=$40)": dict(color="tab:green",  marker="s"),
}

def site_class(site):
    if site == "121_NUCLEAR_1":
        return "Nuclear, solo (ω=0.5, B=$40)"
    return ("Wind, solo (ω=0.5, B=$40)" if site in WIND_BIG
            else "PV + tail, solo (ω=0.5, B=$40)")

fig, ax = plt.subplots(figsize=FIGSIZE)
for cls, style in CLASS_STYLE.items():
    sub = solo[solo["oat_site"].map(site_class) == cls]
    ax.scatter(-sub["delta_curtailment_mwh"] / 1e6,
               sub["delta_cost_less_synthetic_usd_APPROX"] / 1e6,
               s=70, label=cls, zorder=3, edgecolor="black", linewidth=0.5,
               **style)

x_all = -allrow["delta_curtailment_mwh"] / 1e6
y_all = allrow["delta_cost_less_synthetic_usd_APPROX"] / 1e6
x_pil = -pilot["delta_curtailment_mwh"] / 1e6
y_pil = pilot["delta_cost_less_synthetic_usd_APPROX"] / 1e6
ax.scatter([x_all], [y_all], s=260, marker="*", color="tab:red",
           edgecolor="black", linewidth=0.7, zorder=4,
           label="all 11 sites together (ω=0.5, B=$40)")
ax.scatter([x_pil], [y_pil], s=150, marker="D", color="tab:purple",
           edgecolor="black", linewidth=0.7, zorder=4,
           label="pilot portfolio (mid-range ω, B=$25)")

ax.axhline(0, color="grey", linewidth=0.8, zorder=1)
ax.axvline(BASE_CURT_MWH / 1e6, color="grey", linestyle="--", linewidth=1.2, zorder=1)
ax.text(BASE_CURT_MWH / 1e6 - 0.015, 44,
        f"total waste available ({BASE_CURT_MWH/1e6:.3f} TWh/yr)",
        rotation=90, ha="right", va="center", fontsize=11, color="dimgrey")

# (a) nuclear below y = 0
nuc = solo[solo["oat_site"] == "121_NUCLEAR_1"].iloc[0]
ax.annotate("nuclear: more cleanup,\n*lower* system cost",
            xy=(-nuc["delta_curtailment_mwh"] / 1e6,
                nuc["delta_cost_less_synthetic_usd_APPROX"] / 1e6),
            xytext=(0.55, -11), fontsize=11.5,
            arrowprops=dict(arrowstyle="->", linewidth=0.9))
ax.set_ylim(-16, 80)

# (b) the money comparison: pilot vs __ALL__
ax.annotate("", xy=(x_all, y_all), xytext=(x_pil, y_pil),
            arrowprops=dict(arrowstyle="<->", linewidth=1.1))
ax.text((x_all + x_pil) / 2 - 0.31, (y_all + y_pil) / 2,
        f"same cleanup,\n${(y_all - y_pil):.0f}M/yr apart",
        fontsize=12, ha="left", va="center")

ax.set_xlabel("Curtailment removed [TWh/yr]")
ax.set_ylabel("Δ system cost [M$/yr]")
ax.grid(alpha=0.25, linewidth=0.5)
ax.legend(loc="upper left")
save(fig, "fig1_tradeoff")
plt.show()


## P0 · fig2 — "Additive models overpromise by 55%"

Sum of solo effects vs the measured joint run vs the physical cap.
Target slide: 8 (Finding 3) — promotes sub-additivity from anecdote to the
motivation for joint (Bayesian) optimization.


In [ ]:
sum_solo = -solo["delta_curtailment_mwh"].sum() / 1e6      # 2.273 TWh
measured = -allrow["delta_curtailment_mwh"] / 1e6          # 1.471 TWh
cap = BASE_CURT_MWH / 1e6                                  # 1.689 TWh

fig, ax = plt.subplots(figsize=FIGSIZE)
bars = ax.bar([0, 1, 2], [sum_solo, measured, cap], width=0.55,
              color=["tab:blue", "tab:orange", "0.75"],
              edgecolor="black", linewidth=0.7, zorder=3)
ax.axhline(cap, color="black", linestyle="--", linewidth=1.2, zorder=2)

for x, v in zip([0, 1, 2], [sum_solo, measured, cap]):
    ax.text(x, v + 0.04, f"{v:.2f}", ha="center", fontsize=12)

# the -35% gap between promised and measured
gap_pct = (measured - sum_solo) / sum_solo * 100
ax.annotate("", xy=(0.5, measured), xytext=(0.5, sum_solo),
            arrowprops=dict(arrowstyle="<->", linewidth=1.1))
ax.text(0.56, (sum_solo + measured) / 2,
        f"{gap_pct:.0f}%\n(sub-additive)", fontsize=12.5, va="center")

ax.set_xticks([0, 1, 2])
ax.set_xticklabels(["sum of 11\nsolo effects", "measured\nall-together",
                    "physical cap\n(total waste)"], fontsize=12)
ax.set_ylabel("Curtailment removed [TWh/yr]")
ax.set_ylim(0, 2.55)
ax.grid(axis="y", alpha=0.25, linewidth=0.5)
fig.text(0.5, -0.02,
         "sites compete for the same cheap hours → separable/additive "
         "surrogates are structurally wrong → joint optimization",
         ha="center", fontsize=12, style="italic")
save(fig, "fig2_additivity")
plt.show()


## P1 · fig3 — "BO recovers the optimum with a fraction of the grid"

**Truth model (calibrated, honestly labeled):** solo response
`s_i(ω) = A_i (1 - exp(-3ω))` with `A_i` pinned so `s_i(0.5)` equals the
measured solo removal (303: 671,602; 317: 498,712 MWh); joint
`f = s₁ + s₂ - s₁ s₂ / C` with C = 1,688,576 MWh (shared cheap-hours
pool). The shape k = 3 is an assumption. Evaluated on **exactly the 81
grid points of `waves/contour_303x317_A/design_matrix.csv`**, so this cell
is a drop-in replay harness for the measured atlas.

**Replay validity (math-log §5.1):** inside the loop the GP and its output
scaling are fit ONLY on the points queried so far (no leakage), and BO
reports its own seed variance (20 seeds).


In [ ]:
import warnings

from scipy.stats import norm, qmc
from sklearn.exceptions import ConvergenceWarning
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel

warnings.filterwarnings("ignore", category=ConvergenceWarning)

K_SHAPE = 3.0                 # assumed saturation shape (stated on figure)
C_POOL = 1_688_576.0          # shared cheap-hours pool [MWh]
SOLO_05 = {"303": 671_602.0, "317": 498_712.0}   # measured at omega = 0.5
A = {k: v / (1 - np.exp(-K_SHAPE * 0.5)) for k, v in SOLO_05.items()}

def truth(w303, w317):
    s1 = A["303"] * (1 - np.exp(-K_SHAPE * w303))
    s2 = A["317"] * (1 - np.exp(-K_SHAPE * w317))
    return s1 + s2 - s1 * s2 / C_POOL

dm_A = pd.read_csv(CAMP / "waves" / "contour_303x317_A" / "design_matrix.csv")
W = dm_A[["wind_303_omega", "wind_317_omega"]].to_numpy()      # the 81 points
F_SYN = truth(W[:, 0], W[:, 1])
lo, hi = W.min(axis=0), W.max(axis=0)
XN = (W - lo) / (hi - lo)                                      # normalized [0,1]^2
print(f"truth at (0.5, 0.5): {truth(0.5, 0.5)/1e6:.3f} TWh "
      f"(pair sub-additivity {truth(0.5,0.5)/(SOLO_05['303']+SOLO_05['317'])*100-100:.0f}%)")

N_EVALS, N_INIT, N_SEEDS, N_GRID = 30, 5, 20, len(W)

def replay_bo(f_grid, seed):
    rng = np.random.default_rng(seed)
    idx = list(rng.choice(N_GRID, N_INIT, replace=False))
    for _ in range(N_INIT, N_EVALS):
        X, y = XN[idx], f_grid[idx]
        gp = GaussianProcessRegressor(
            kernel=Matern(length_scale=0.3, nu=2.5)
                   + WhiteKernel(1e-6, noise_level_bounds=(1e-12, 1e-2)),
            normalize_y=True, n_restarts_optimizer=2, random_state=seed)
        gp.fit(X, y)          # fit ONLY on queried points -- no leakage
        cand = np.setdiff1d(np.arange(N_GRID), idx)
        mu, sd = gp.predict(XN[cand], return_std=True)
        sd = np.maximum(sd, 1e-12)
        z = (mu - y.max()) / sd
        ei = (mu - y.max()) * norm.cdf(z) + sd * norm.pdf(z)
        idx.append(int(cand[np.argmax(ei)]))
    return idx

def replay_random(f_grid, seed):
    return list(np.random.default_rng(seed).permutation(N_GRID)[:N_EVALS])

def replay_sobol(f_grid, seed):
    # draw a power-of-2 block (Sobol balance, no scipy warning), keep 30
    pts = qmc.Sobol(2, scramble=True, seed=seed).random(32)[:N_EVALS]
    idx, used = [], np.zeros(N_GRID, bool)
    for p in pts:                       # nearest unused grid point, in sequence
        d = np.linalg.norm(XN - p, axis=1)
        d[used] = np.inf
        j = int(np.argmin(d))
        idx.append(j); used[j] = True
    return idx

def best_so_far(f_grid, idx):
    return np.maximum.accumulate(f_grid[idx]) / f_grid.max()

def run_replay(f_grid):
    out = {}
    for name, fn in [("BO (GP + EI)", replay_bo),
                     ("random search", replay_random),
                     ("Sobol sequence", replay_sobol)]:
        out[name] = np.array([best_so_far(f_grid, fn(f_grid, s))
                              for s in range(N_SEEDS)])
    return out

def evals_to(frac, curves):
    firsts = [np.argmax(c >= frac) + 1 if (c >= frac).any() else np.nan
              for c in curves]
    return float(np.nanmedian(firsts))

curves_syn = run_replay(F_SYN)
BO_99 = evals_to(0.99, curves_syn["BO (GP + EI)"])
print(f"median evals to 99% of grid optimum -- BO: {BO_99:.0f}, "
      f"random: {evals_to(0.99, curves_syn['random search']):.0f}, "
      f"Sobol: {evals_to(0.99, curves_syn['Sobol sequence']):.0f}, "
      f"enumeration: {N_GRID}")


In [ ]:
METHOD_COLORS = {"BO (GP + EI)": "tab:blue", "random search": "tab:orange",
                 "Sobol sequence": "tab:green"}
g303 = np.unique(W[:, 0]); g317 = np.unique(W[:, 1])

def plot_bo_figure(f_grid, curves, fname, banner):
    fig, (axL, axR) = plt.subplots(1, 2, figsize=FIGSIZE)

    # (a) truth surface + one representative BO run (seed 0)
    Z = f_grid.reshape(len(g303), len(g317)).T / 1e6   # rows = omega_317
    cs = axL.contourf(g303, g317, Z, levels=14, cmap="viridis")
    cbar = fig.colorbar(cs, ax=axL, pad=0.02)
    cbar.set_label("Curtailment removed [TWh/yr]", fontsize=12, fontweight="bold")
    seq = replay_bo(f_grid, seed=0)
    PT = 0.004  # ~data units per offset point (for label collision checks)
    anchors = []
    for t, j in enumerate(seq, start=1):
        x, y = W[j]
        if t <= N_INIT:
            axL.scatter(x, y, s=70, facecolor="none", edgecolor="black",
                        linewidth=1.4, zorder=4)
        else:
            axL.scatter(x, y, s=55, color="tab:red", edgecolor="white",
                        linewidth=0.8, zorder=4)
            # number offsets point INTO the axes at the grid edges; nudge
            # down until clear of previously placed labels
            dx = -13 if x > 0.93 else 5
            dy = -11 if y > 0.93 else 4
            for _ in range(8):
                ax_, ay_ = x + dx * PT, y + dy * PT
                if all(abs(ax_ - a) > 0.055 or abs(ay_ - b) > 0.035
                       for a, b in anchors):
                    break
                dy -= 12
            anchors.append((x + dx * PT, y + dy * PT))
            axL.annotate(str(t), (x, y), textcoords="offset points",
                         xytext=(dx, dy), fontsize=8.5, color="black")
    axL.scatter([], [], s=70, facecolor="none", edgecolor="grey",
                linewidth=1.4, label="BO init (5 random)")
    axL.scatter([], [], s=55, color="tab:red", label="BO queries (numbered)")
    axL.legend(loc="lower left", fontsize=10.5, framealpha=0.95)
    axL.set_xlabel("PEM size fraction ω (303_WIND)")
    axL.set_ylabel("PEM size fraction ω (317_WIND)")
    axL.text(0.02, 1.02, "(a)", transform=axL.transAxes, fontsize=13,
             fontweight="bold")

    # (b) best-so-far recovery, median + IQR over seeds
    ev = np.arange(1, N_EVALS + 1)
    for name, arr in curves.items():
        med = np.median(arr, axis=0)
        q1, q3 = np.percentile(arr, [25, 75], axis=0)
        axR.plot(ev, med, color=METHOD_COLORS[name], linewidth=2, label=name)
        axR.fill_between(ev, q1, q3, color=METHOD_COLORS[name], alpha=0.18)
    axR.axhline(1.0, color="black", linestyle="--", linewidth=1)
    axR.text(N_EVALS, 1.002, f"grid optimum ({N_GRID} evals)", ha="right",
             va="bottom", fontsize=10.5)
    axR.set_xlabel("Evaluations (full-year PCM runs)")
    axR.set_ylabel("Fraction of grid optimum")
    axR.set_ylim(0.80, 1.015)
    axR.set_xlim(1, N_EVALS)
    axR.grid(alpha=0.25, linewidth=0.5)
    axR.legend(loc="lower right")
    axR.text(0.02, 1.02, "(b)", transform=axR.transAxes, fontsize=13,
             fontweight="bold")

    fig.text(0.5, -0.04, banner, ha="center", fontsize=11.5, style="italic")
    fig.tight_layout()
    save(fig, fname)
    plt.show()

SYN_BANNER = ("illustrative landscape calibrated to screening measurements "
              "(s(0.5) pinned, k = 3 shape assumed) — replaced by the "
              "measured 81-run atlas when it lands")
plot_bo_figure(F_SYN, curves_syn, "fig3_bo_demo", SYN_BANNER)
plot_bo_figure(F_SYN, curves_syn, "fig3_bo_demo_synthetic", SYN_BANNER)
FIG3_SOURCE = "synthetic"


## P1 · fig4 — two-regime noise made visible

Left: full-year repeats are bit-identical (deterministic pipeline).
Right: MILP optimality-gap *path noise* — five near-twin small designs
scatter ±2.3 GWh across zero (sign flip), while the big wind sites sit
far outside the ±3 GWh trust floor. Target slide: 6.


In [ ]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=FIGSIZE,
                               gridspec_kw={"width_ratios": [1, 1.5]})

# (a) determinism: the 3 pilot full-year repeats
rep = PILOT_REPEAT_COST / 1e6
axL.scatter([1, 2, 3], [rep] * 3, s=110, color="tab:blue",
            edgecolor="black", linewidth=0.7, zorder=3)
axL.set_xticks([1, 2, 3])
axL.set_xlim(0.4, 3.6)
axL.set_ylim(rep - 1.5, rep + 1.5)
axL.set_xlabel("Pilot repeat run")
axL.set_ylabel("Total system cost [M$/yr]")
axL.annotate(f"${PILOT_REPEAT_COST:,.2f}\n× 3, bit-identical",
             xy=(2, rep), xytext=(0.62, rep + 0.75), fontsize=11.5,
             arrowprops=dict(arrowstyle="->", linewidth=0.9))
axL.grid(alpha=0.25, linewidth=0.5)
axL.text(0.02, 1.02, "(a)", transform=axL.transAxes, fontsize=13,
         fontweight="bold")

# (b) Delta shed: five near-twin small designs vs the big wind sites
SMALL = ["324_PV_1", "324_PV_2", "324_PV_3", "310_PV_2", "320_PV_1"]
BIG = ["303_WIND_1", "317_WIND_1", "122_WIND_1"]
shed = solo.set_index("oat_site")["delta_load_shed_mwh"] / 1e3   # GWh
rng = np.random.default_rng(1)
xs_small = 0 + rng.uniform(-0.13, 0.13, len(SMALL))
xs_big = 1 + rng.uniform(-0.13, 0.13, len(BIG))
axR.axhspan(-3, 3, color="tab:grey", alpha=0.15, zorder=1)
axR.text(1.38, 2.55, "±3 GWh trust floor", fontsize=10.5,
         ha="right", color="dimgrey")
axR.axhline(0, color="grey", linewidth=0.8, zorder=1)
axR.scatter(xs_small, shed[SMALL], s=90, color="tab:orange", marker="o",
            edgecolor="black", linewidth=0.6, zorder=3,
            label="five near-twin small designs")
axR.scatter(xs_big, shed[BIG], s=100, color="tab:blue", marker="s",
            edgecolor="black", linewidth=0.6, zorder=3,
            label="big wind sites")
flip = shed["324_PV_3"]
axR.annotate("sign flips across\nnear-twin designs",
             xy=(xs_small[2], flip), xytext=(-0.42, -9.5), fontsize=11,
             arrowprops=dict(arrowstyle="->", linewidth=0.9))
axR.set_xticks([0, 1])
axR.set_xticklabels(["small PV / tail solos\n(near-identical designs)",
                     "big wind solos"], fontsize=12)
axR.set_xlim(-0.55, 1.45)
axR.set_ylabel("Δ load shed [GWh/yr]")
axR.grid(axis="y", alpha=0.25, linewidth=0.5)
axR.legend(loc="lower left")
axR.text(0.02, 1.02, "(b)", transform=axR.transAxes, fontsize=13,
         fontweight="bold")

fig.tight_layout()
save(fig, "fig4_noise")
plt.show()


## P2 · real-data auto-upgrade (contour_303x317_A) — auto-detected, never required


In [ ]:
FIG5_STATUS = "absent"
obj_A_path = CAMP / "waves" / "contour_303x317_A" / "objectives.csv"
if obj_A_path.is_file():
    obj_A = pd.read_csv(obj_A_path)
    obj_A = obj_A[obj_A["num_days"] == 366]
    n_A = len(obj_A)
    if n_A >= 77:
        # rebuild fig3 on the MEASURED truth table (synthetic copy already
        # saved); align by design `index`, NOT dataframe position
        val = dict(zip(obj_A["index"], -obj_A["delta_curtailment_mwh"]))
        f_meas = np.array([val.get(int(i), np.nan) for i in dm_A["index"]])
        miss = np.isnan(f_meas)
        if miss.any():   # fill the few missing cells from the calibrated model
            f_meas[miss] = truth(W[miss, 0], W[miss, 1])
        curves_meas = run_replay(f_meas)
        BO_99 = evals_to(0.99, curves_meas["BO (GP + EI)"])
        plot_bo_figure(f_meas, curves_meas, "fig3_bo_demo",
                       f"measured ground truth: the {n_A}-run "
                       "contour_303x317_A atlas (ρ = $1/kg)")
        FIG3_SOURCE = f"measured ({n_A}/81 runs)"
        print(f"fig3 UPGRADED to measured truth; BO evals to 99%: {BO_99:.0f}")
    elif n_A >= 30:
        merged = dm_A.merge(obj_A[["index", "delta_curtailment_mwh"]], on="index")
        Zp = np.full((len(g303), len(g317)), np.nan)
        for _, r in merged.iterrows():
            i = int(np.argmin(np.abs(g303 - r["wind_303_omega"])))
            j = int(np.argmin(np.abs(g317 - r["wind_317_omega"])))
            Zp[i, j] = -r["delta_curtailment_mwh"] / 1e6
        import matplotlib as mpl
        fig, ax = plt.subplots(figsize=(7.2, 5.6))
        cmap = mpl.colormaps["viridis"].copy()
        cmap.set_bad("0.85")
        pm = ax.pcolormesh(g303, g317, np.ma.masked_invalid(Zp).T,
                           cmap=cmap, shading="nearest")
        cbar = fig.colorbar(pm, ax=ax, pad=0.02)
        cbar.set_label("Curtailment removed [TWh/yr]", fontsize=12,
                       fontweight="bold")
        ax.set_xlabel("PEM size fraction ω (303_WIND)")
        ax.set_ylabel("PEM size fraction ω (317_WIND)")
        ax.set_title(f"as of this morning, {n_A}/81 runs", fontsize=13)
        save(fig, "fig5_contour_partial")
        plt.show()
        FIG5_STATUS = f"partial ({n_A}/81)"
    else:
        print(f"contour_A objectives present but only {n_A} rows -- skipping")
else:
    print("contour_303x317_A/objectives.csv not present -- fig3 stays synthetic")


## P2 · placebo verdict hook — auto-detected, never required


In [ ]:
import subprocess
import sys

PLACEBO_STATUS = "not available yet"
if (CAMP / "waves" / "placebo" / "objectives.csv").is_file():
    res = subprocess.run([sys.executable, "analyze_placebo.py"], cwd=CAMP,
                         capture_output=True, text=True)
    (FIGS / "placebo_verdict.txt").write_text(res.stdout + res.stderr)
    PLACEBO_STATUS = "verdict saved to figs/placebo_verdict.txt"
    print(res.stdout)
else:
    print("waves/placebo/objectives.csv not present -- skipping placebo verdict")


## P2 · fig6 (stretch) — bi-objective flavor on the synthetic truth

Linear per-site cost pinned to the measured solo Δcost at ω = 0.5
(303: +$9.28M, 317: +$13.24M; no interaction) — the (cost, removal) image
of the 81 grid points with the Pareto front highlighted.


In [ ]:
COST_05 = {"303": 9.28, "317": 13.24}   # M$/yr at omega = 0.5, measured solo
cost81 = (W[:, 0] / 0.5) * COST_05["303"] + (W[:, 1] / 0.5) * COST_05["317"]
rem81 = F_SYN / 1e6

order = np.argsort(cost81)
pareto, best = [], -np.inf
for j in order:                        # min cost, max removal
    if rem81[j] > best:
        pareto.append(j); best = rem81[j]
pareto = np.array(pareto)

fig, ax = plt.subplots(figsize=(7.5, 5.6))
ax.scatter(cost81, rem81, s=45, color="0.65", edgecolor="black",
           linewidth=0.4, zorder=2, label="81 grid designs")
ax.plot(cost81[pareto], rem81[pareto], "-o", color="tab:red", linewidth=2,
        markersize=7, markeredgecolor="black", markeredgewidth=0.6, zorder=3,
        label="Pareto front (min cost, max removal)")
ax.set_xlabel("Δ system cost [M$/yr]")
ax.set_ylabel("Curtailment removed [TWh/yr]")
ax.grid(alpha=0.25, linewidth=0.5)
ax.legend(loc="lower right")
fig.text(0.5, -0.03,
         "synthetic truth + linear cost model (no cost interaction) — "
         "illustrative only",
         ha="center", fontsize=11, style="italic")
save(fig, "fig6_biobjective")
plt.show()


## FIGURES.md — regenerated on every run (captions stay in sync with data)


In [ ]:
figures_md = f'''# Group-meeting figures — generated {pd.Timestamp.now():%Y-%m-%d %H:%M}

All figures: 300 dpi PNG + PDF in `figs/`, journal style, produced by
`gm_figures.ipynb` (this file is regenerated by the notebook's last cell).

## fig1_tradeoff.png
- **Caption:** Every full-year design measured so far on the (curtailment
  removed, Δ system cost) plane: cleanup is not free — except at the
  nuclear site, whose retrofit removes 0.37 TWh/yr while *lowering* system
  cost — and the pilot portfolio matches the all-sites-on design's cleanup
  (1.50 vs 1.47 TWh/yr) at ${y_all - y_pil:.0f}M/yr lower cost: design sets
  the price.
- **Target slide:** right after the −89% hook (slide 2/3).
- **Caveats:** cost is the synthetic-bid-corrected APPROXIMATION (base-nuclear
  fuel asymmetry pending); the pilot and __ALL__ designs differ in both sizes
  AND bid ($25 vs $40), so the $49M gap is not a pure sizing effect.

## fig2_additivity.png
- **Caption:** Adding up the 11 solo retrofit effects promises
  {sum_solo:.2f} TWh/yr of curtailment removal — 35% more than the measured
  all-together run ({measured:.2f} TWh/yr) and beyond the {cap:.3f} TWh/yr
  physically available: sites compete for the same cheap hours, so additive
  reasoning is structurally wrong and sizing must be optimized jointly.
- **Target slide:** 8 (Finding 3), as the motivation for BO.
- **Caveats:** none beyond measurement noise (Δcurt floor ≈ 5k MWh).

## fig3_bo_demo.png  (truth source this run: {FIG3_SOURCE})
- **Caption:** Bayesian optimization replayed on the 81-point
  (ω_303, ω_317) design grid: the GP + Expected-Improvement loop reaches
  99% of the grid optimum in a median of {BO_99:.0f} full-year PCM
  evaluations vs 81 by enumeration ({BO_99/81*100:.0f}% of the budget),
  ahead of random and Sobol baselines (20 seeds, median ± IQR).
- **Target slide:** the BO forward-path slide.
- **Caveats:** the landscape is illustrative — calibrated so each site's solo
  response matches its measured removal at ω = 0.5 and coupled through the
  shared cheap-hours pool; the saturation shape k = 3 is an assumption. The
  synthetic version is always kept at `fig3_bo_demo_synthetic.png`; the main
  file upgrades automatically to the measured atlas when
  `waves/contour_303x317_A/objectives.csv` lands. No leakage: the GP is fit
  only on queried points.

## fig4_noise.png
- **Caption:** The pipeline's two-regime noise: full-year repeats are
  bit-identical (left, ${PILOT_REPEAT_COST:,.2f} × 3), yet five nearly
  identical small-PV/tail designs scatter ±2.3 GWh/yr in Δ load shed with a
  sign flip (right) — MILP optimality-gap path noise — while the big wind
  sites sit far outside the ±3 GWh trust floor: effects inside the band are
  reported as "0 within noise", never as findings.
- **Target slide:** 6 (currently text-only).
- **Caveats:** wind-site Δshed carries the f4 DA-commitment caveat until the
  placebo run lands (status: {PLACEBO_STATUS}).

## fig5_contour_partial.png  (status this run: {FIG5_STATUS})
- Only produced when 30–76 contour_A rows are available: measured partial
  contour, missing cells grey — "as of this morning" live-data slide.

## fig6_biobjective.png
- **Caption:** The 81 grid designs in (cost, removal) space with the Pareto
  front highlighted — sizing is a genuine trade-off, and BO's multi-objective
  form targets exactly this front.
- **Target slide:** optional companion to the BO slide.
- **Caveats:** synthetic truth + linear cost (no cost interaction);
  illustrative only.

## Morning-of checklist
1. If CRC results landed overnight, put them where the auto-upgrade looks:
   on CRC run `python summarize_wave.py waves/contour_303x317_A` (and
   `waves/placebo`), commit the objectives.csv files, then `git pull` here.
2. Re-run headlessly and refresh every figure + this file:
   ```
   source ~/venvs/gm0829/bin/activate
   cd trial_0826/campaign/analysis/gm_0829
   jupyter nbconvert --to notebook --execute gm_figures.ipynb --output gm_figures.executed.ipynb
   ```
   ≥ 77 contour rows → fig3 becomes measured ground truth; 30–76 →
   fig5_contour_partial.png appears; placebo present →
   figs/placebo_verdict.txt.
'''
(GM / "FIGURES.md").write_text(figures_md)
print("wrote FIGURES.md")
